# Lab 03B: Create or Select the Reference Agent

**Duration:** 10-15 minutes. Complete this setup before [Lab 04](../lab-04-portal-sdk-custom/lab-04-portal-sdk-custom.ipynb).

Select an existing Contoso agent, or explicitly opt in to creating a minimal prompt agent when none exists. This is setup, not an evaluation run. You need an existing Foundry project, model deployment, and `az login` authentication; creating an agent also requires write permission.

The new agent uses the instructions in the [reference configuration](../../src/reference-agent/agent-config.yaml). It has **no retrieval or executable tools**, and does not receive the golden answers. Existing agents are never modified. [Lab 05](../lab-05-rag-evaluation/lab-05-rag-evaluation.ipynb) explores simulated RAG configurations; [Lab 06](../lab-06-agent-evaluation/lab-06-agent-evaluation.ipynb) creates a separate tool-enabled evaluation target. These targets are not interchangeable baselines.

Keep the resolved name and version pinned for Lab 04 and optional [Lab 06B: advanced judge calibration](../lab-06-agent-evaluation/lab-06b-llm-as-judge-rubric.ipynb). Cloud resource creation and subsequent model calls may incur charges.

In [ ]:
import os
from pathlib import Path

import yaml
from dotenv import load_dotenv, set_key
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.core.exceptions import ResourceNotFoundError

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file())
env_path = repo_root / '.env'
load_dotenv(env_path)
project_endpoint = os.environ.get('AI_FOUNDRY_PROJECT_ENDPOINT')
tenant_id = os.environ.get('TENANT_ID')
model_deployment = os.environ.get('AZURE_AI_MODEL_DEPLOYMENT_NAME', 'gpt-4o')
agent_name = os.environ.get('AI_FOUNDRY_AGENT_NAME', 'contoso-support-agent-eval')
agent_version = os.environ.get('AI_FOUNDRY_AGENT_VERSION')

CREATE_IF_MISSING = False  # Explicit opt-in; never adds a version to an existing agent.
SAVE_SELECTION = False  # Opt in to updating only the agent name/version in the local .env.

if not project_endpoint:
    raise ValueError('AI_FOUNDRY_PROJECT_ENDPOINT is required. Configure the project in .env first.')
reference_config = yaml.safe_load((repo_root / 'src' / 'reference-agent' / 'agent-config.yaml').read_text(encoding='utf-8'))
project_client = AIProjectClient(endpoint=project_endpoint, credential=AzureCliCredential(tenant_id=tenant_id))

In [ ]:
def select_reference_agent(project_client, name, version, model, instructions, create_if_missing=False):
    try:
        if version:
            return project_client.agents.get_version(agent_name=name, agent_version=version)
        return project_client.agents.get(agent_name=name).versions.latest
    except ResourceNotFoundError as exc:
        if version or not create_if_missing:
            raise RuntimeError(
                f'Agent {name!r} (version {version or "latest"}) was not found. '
                'Check the configured name/version. To create a missing agent, clear '
                'AI_FOUNDRY_AGENT_VERSION and explicitly set CREATE_IF_MISSING=True.'
            ) from exc
    return project_client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(model=model, instructions=instructions),
    )

agent = select_reference_agent(
    project_client, agent_name, agent_version, model_deployment,
    reference_config['instructions'], CREATE_IF_MISSING,
)
print(f'Reference agent: {agent.name}, version {agent.version}')

## Pin the selection across notebook kernels

Review the name/version above. Set `SAVE_SELECTION=True` and rerun the next cell to save both values in the Git-ignored root `.env`, preserving other settings. Otherwise copy its printed assignments into that file yourself. A Python variable in this notebook does not configure another notebook's kernel.

Process environment variables take precedence over `.env`. Remove or update any conflicting process-level agent settings, and restart already-running Lab 04/06B kernels after changing the file. To deliberately evaluate another version later, update the pin and produce a new baseline; do not silently compare different targets.

In [ ]:
if SAVE_SELECTION:
    set_key(str(env_path), 'AI_FOUNDRY_AGENT_NAME', agent.name)
    set_key(str(env_path), 'AI_FOUNDRY_AGENT_VERSION', str(agent.version))
    print(f'Saved the reference agent name/version to {env_path}.')
else:
    print('Selection not saved. Add these settings to the root .env before continuing:')
print(f'AI_FOUNDRY_AGENT_NAME={agent.name}')
print(f'AI_FOUNDRY_AGENT_VERSION={agent.version}')

## Next steps

Continue with [Lab 04: basic evaluation](../lab-04-portal-sdk-custom/lab-04-portal-sdk-custom.ipynb), then [Lab 05: RAG evaluation](../lab-05-rag-evaluation/lab-05-rag-evaluation.ipynb), [Lab 06: tool-trajectory evaluation](../lab-06-agent-evaluation/lab-06-agent-evaluation.ipynb), and optional [Lab 06B: advanced judge calibration](../lab-06-agent-evaluation/lab-06b-llm-as-judge-rubric.ipynb).